# Predição: Prova de Fogo - V2

Autor:  Viviane da Rosa Sommer

Atualizado: 29/11/2024

## Objetivo:
Notebook para fazer a predição do modelo de Coral-Sol (MSO-V1) no vídeo da prova de fogo - V1.
Essa versão gerar um vídeo com os contornos encima do vídeo original, junto de suas probabilidades.
Ao final, também é gerado um gráfico e CSV , relacionado aos Scores (Probabilidade Média dos Contornos) vs Tempo.

## Importações Necessárias

In [ ]:
!pip install ultralytics
!pip install opencv-python-headless
!pip install torch
!pip install tqdm
!pip install pandas

import cv2
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
import torchvision
import torch
from tqdm.notebook import tqdm
import pandas as pd
from typing import List

## Declaração de Constantes e Modelos

In [ ]:
CORAL_CLASS_ID = 0
RESIZED_DIM_CORAL = 800
coral_model = YOLO('runs/segment/Lote123-SOverlay/train-800-lote123/weights/best.pt')

## Funções Necessárias

In [ ]:
def process_results(results: list) -> any:
    """
    Processes the results obtained from the model, returning the first valid result with masks.

    Parameters:
        results (list): List of model detection results.

    Returns:
        any: The first valid result containing masks, or None if no valid result is found.
    """
    for result in results:
        if result.masks is not None:
            return result
    return None

def prediction_coral(img: np.ndarray) -> tuple:
    """
    Predicts coral objects using the coral model and returns individual masks and their scores.

    Parameters:
        img (np.ndarray): Input image for prediction.

    Returns:
        tuple: A tuple containing:
            - masks (list): List of binary masks for the predicted coral regions.
            - scores (list): List of scores for each predicted coral region.
    """
    original_height, original_width = img.shape[:2]
    coral_results = coral_model(img, verbose=False)
    coral_best_result = process_results(coral_results)
    
    masks_list = []
    scores_list = []
    
    if coral_best_result is not None and coral_best_result.masks is not None:
        masks = coral_best_result.masks.data
        boxes = coral_best_result.boxes.data
        classes = boxes[:, 5]
        scores = boxes[:, 4]

        coral_indices = torch.where((classes == CORAL_CLASS_ID) & (scores > 0.5))[0]
        coral_boxes = boxes[coral_indices]
        coral_masks = masks[coral_indices]
        coral_scores = scores[coral_indices]

        if len(coral_boxes) > 0:
            nms_indices = torchvision.ops.nms(coral_boxes[:, :4], coral_scores, iou_threshold=0.2)
            coral_boxes = coral_boxes[nms_indices]
            coral_masks = coral_masks[nms_indices]
            coral_scores = coral_scores[nms_indices]
            
            for mask in coral_masks:
                mask_np = mask.int().cpu().numpy() * 255
                if mask_np.dtype != np.uint8:
                    mask_np = mask_np.astype(np.uint8)
                masks_list.append(mask_np)
            scores_list = coral_scores.tolist()
    
    return masks_list, scores_list

def process_video_and_save_scores(video_path: str, output_path: str) -> tuple:
    """
    Process the video frame by frame, perform inference and generate a video with the predictions.
    This function also saves the times and average scores.

    Parameters:
        video_path (str): Path to the input video file.
        output_path (str): Path to the output video file.

    Returns:
        tuple: Two lists, one with times and one with average scores per frame.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Error opening video.")
        return
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    
    times = []
    average_scores = []
    frame_count = 0
    
    for _ in tqdm(range(total_frames), desc="Processing Video"):
        ret, frame = cap.read()
        if not ret:
            break

        timestamp = frame_count / fps
        times.append(timestamp)
        
        cropped_frame, top, bottom = crop_frame(frame)
        masks_list, scores_list = prediction_coral(cropped_frame)
        frame_with_mask = overlay_masks(frame, masks_list, top, bottom, scores_list)
        
        if scores_list:
            average_score = sum(scores_list) / len(scores_list)
        else:
            average_score = 0.0
        average_scores.append(average_score)
        
        out.write(frame_with_mask)
        frame_count += 1
    
    cap.release()
    out.release()
    
    return times, average_scores


def crop_frame(frame: np.ndarray) -> tuple:
    """
    Crop the frame to focus on the central region.

    Parameters:
        frame (np.ndarray): The frame to crop.

    Returns:
        tuple: A tuple containing the cropped frame and the top and bottom crop coordinates.
    """
    height, width, _ = frame.shape
    mid = height // 2
    top = max(0, mid - int(0.34 * height))
    bottom = min(height, mid + int(0.34 * height))
    cropped_frame = frame[top:bottom, :]
    return cropped_frame, top, bottom

def overlay_masks(frame: np.ndarray, masks_list: list, top: int, bottom: int, scores_list: list) -> np.ndarray:
    """
    Overlay the masks on the original frame and add annotations.

    Parameters:
        frame (np.ndarray): The original video frame.
        masks_list (list): List of binary masks to overlay.
        top (int): The top crop coordinate.
        bottom (int): The bottom crop coordinate.
        scores_list (list): List of scores for each mask.

    Returns:
        np.ndarray: The frame with the masks overlaid and annotations.
    """
    for mask, score in zip(masks_list, scores_list):
        mask_resized = cv2.resize(mask, (frame.shape[1], bottom - top))
        contours, _ = cv2.findContours(mask_resized, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(frame[top:bottom, :], contours, -1, (0, 0, 255), 2)  # Contornos vermelhos
        for contour in contours:
            x, y, w, h = cv2.boundingRect(contour)
            score_text = f'{score:.2f}'
            cv2.putText(frame, score_text, (x, y + top - 5), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 1, cv2.LINE_AA)  # Texto branco

    return frame

def plot_scores(times: List[float], scores: List[float]) -> None:
    """
    Plots the scores over time.

    Parameters:
        times (List[float]): List of times corresponding to the frames.
        scores (List[float]): List of scores for each frame.
        
    Returns:
        None
    """
    plt.figure(figsize=(12, 6))
    plt.plot(times, scores, marker='o', linestyle='-', color='b')
    plt.xlabel('Time (seconds)')
    plt.ylabel('Average Score')
    plt.title('Average Score Over Time')
    plt.grid(True)
    plt.show()

def save_scores_to_csv(times: List[float], scores: List[float], csv_path: str) -> None:
    """
    Saves the scores and times to a CSV file.

    Parameters:
        times (List[float]): List of times corresponding to the frames.
        scores (List[float]): List of scores for each frame.
        csv_path (str): Path to the output CSV file.
        
    Returns:
        None
    """
    df = pd.DataFrame({'Time (seconds)': times, 'Average Score': scores})
    df.to_csv(csv_path, index=False)

## Prova de Fogo - V2 - Vídeo do resultado com máscaras

In [ ]:
video_path = 'Prova de fogo 1.mp4'
output_path = 'output_video_v2.mp4'
times, average_scores = process_video_and_save_scores(video_path, output_path)

plot_scores(times, average_scores)

## CSV com o Score vs Tempo, de cada frame do vídeo

In [ ]:
csv_path = 'scores_vs_time.csv'
save_scores_to_csv(times, average_scores, csv_path)